# Extraction JSON — corpus `pdf_SG/`

Notebook unifié : extraction **synchrone** (1 PDF à la fois) ou **batch asynchrone** (API Message Batches, ~50 % moins cher).

**Choisir le mode en cellule suivante**, puis exécuter les cellules dans l'ordre.

| Mode | Usage recommandé |
|------|------------------|
| `sync` | Validation du prompt, petits volumes, résultat immédiat |
| `batch` | Production, grands volumes ; la machine peut être éteinte pendant le traitement |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHOIX DU MODE — modifier uniquement cette valeur avant de lancer       ║
# ║                                                                          ║
# ║   "sync"  : extraction séquentielle, 1 PDF à la fois, résultat immédiat ║
# ║   "batch" : lot asynchrone Anthropic Message Batches (~50 % moins cher)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

EXTRACTION_MODE = "sync"   # <── changer en "batch" pour le mode asynchrone

assert EXTRACTION_MODE in ("sync", "batch"), "EXTRACTION_MODE doit être 'sync' ou 'batch'"
print(f"Mode sélectionné : {EXTRACTION_MODE}")

In [ ]:
# ── Imports & configuration ───────────────────────────────────────────────────
import base64
import json
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import anthropic
import pandas as pd
import pdfplumber
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

# Racine du projet : fonctionne que Jupyter soit lancé depuis la racine
# ou directement depuis le dossier notebooks/.
_here = Path(".").resolve()
BASE_DIR = _here.parent if _here.name == "notebooks" else _here

PDF_DIR            = BASE_DIR / "data" / "pdf_SG"
JSON_DIR           = BASE_DIR / "data" / "json_SG"
PROMPT_PATH        = BASE_DIR / "config" / "extraction_prompt.txt"
DOTENV_PATH        = BASE_DIR / ".env"
BATCH_STATE_PATH   = BASE_DIR / "state" / "batch_pending_extractions.json"
LAST_BATCH_ID_FILE = BASE_DIR / "state" / "batch_last_submitted_id.txt"

JSON_DIR.mkdir(exist_ok=True)


def load_dotenv(path):
    if not Path(path).exists():
        return
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k and k not in os.environ:
                os.environ[k] = v


load_dotenv(DOTENV_PATH)
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
MODEL   = os.environ.get("SG_MODEL", "claude-haiku-4-5-20251001")

assert API_KEY, "ANTHROPIC_API_KEY non défini — ajoute-la dans .env ou via export"

# ── Paramètres (communs et spécifiques au mode) ───────────────────────────────
MAX_DOCS           = int(os.environ.get("SG_MAX_EXTRACTIONS", "0"))  # 0 = illimité (sync)
NB_PDF_IN_BATCH    = 300       # nombre max de PDF par soumission (batch)
ISIN_PREFIXES      = ["XS", "FR"]  # [] = tous les ISIN
POLL_INTERVAL_SEC  = 300       # secondes entre deux polls de statut (batch)
BATCH_CUSTOM_ID_RE = re.compile(r"^[a-zA-Z0-9_-]{1,64}$")

print(f"Modèle            : {MODEL}")
print(f"Mode              : {EXTRACTION_MODE}")
if EXTRACTION_MODE == "sync":
    print(f"MAX_DOCS          : {MAX_DOCS or 'illimité'}")
else:
    print(f"NB_PDF_IN_BATCH   : {NB_PDF_IN_BATCH}")
    print(f"POLL_INTERVAL_SEC : {POLL_INTERVAL_SEC}")

In [ ]:
# ── Helpers état batch ────────────────────────────────────────────────────────
def load_batch_state() -> dict:
    if not BATCH_STATE_PATH.exists():
        return {"pending_stems": [], "jobs": []}
    try:
        return json.loads(BATCH_STATE_PATH.read_text(encoding="utf-8"))
    except Exception:
        return {"pending_stems": [], "jobs": []}


def save_batch_state(state: dict) -> None:
    BATCH_STATE_PATH.write_text(
        json.dumps(state, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


# ── Scan des PDFs éligibles ───────────────────────────────────────────────────
# Exclusions : JSON déjà présent dans json_SG/ OU stem déjà soumis en batch.
extracted_stems    = set(p.stem for p in JSON_DIR.glob("*.json")) if JSON_DIR.exists() else set()
pending_batch_stems = set(load_batch_state().get("pending_stems", []))

limit = NB_PDF_IN_BATCH if EXTRACTION_MODE == "batch" else (MAX_DOCS or None)

seen_pdf_stems = set()
SAMPLE = []
for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    stem = pdf_path.stem
    if stem in seen_pdf_stems:
        continue
    seen_pdf_stems.add(stem)
    if stem in extracted_stems or stem in pending_batch_stems:
        continue
    if ISIN_PREFIXES and not any(stem.upper().startswith(p) for p in ISIN_PREFIXES):
        continue
    SAMPLE.append({"file": pdf_path.name, "path": str(pdf_path), "n_pages": None})
    if limit and len(SAMPLE) >= limit:
        break

print(f"PDFs dans pdf_SG       : {len(list(PDF_DIR.glob('*.pdf')))}")
print(f"JSON déjà présents     : {len(extracted_stems)}")
print(f"Pending batch stems    : {len(pending_batch_stems)}")
print(f"Sélectionnés ici       : {len(SAMPLE)}")
pd.DataFrame(SAMPLE)[["file", "n_pages"]].head(20)

In [ ]:
# ── Helpers : import depuis sg_extract_json (source unique de vérité) ─────────
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from sg_extract_json import (
    max_tokens_for_pages,
    normalize_model_json_response,
    validate_json_output,
)


def pdf_to_base64(path: str) -> str:
    with open(path, "rb") as f:
        return base64.standard_b64encode(f.read()).decode()


SYSTEM_PROMPT = PROMPT_PATH.read_text(encoding="utf-8")
print(f"Prompt chargé : {len(SYSTEM_PROMPT)} caractères")

## Extraction

- **`sync`** : traite les PDFs un par un, résultats écrits dans `json_SG/` immédiatement. Les cellules d'analyse ci-dessous sont exploitables dès la fin.
- **`batch`** : soumet le lot à l'API Anthropic, puis tu peux éteindre la machine. Exécute la cellule **Poll + import** plus bas pour récupérer les résultats.

In [ ]:
# ── Extraction : mode sync ou soumission batch ────────────────────────────────
client  = anthropic.Anthropic(api_key=API_KEY)
RESULTS = {}  # rempli uniquement en mode sync (utilisé par les cellules d'analyse)

# ════════════════════════════════════════════════════════════════════════════
#  MODE SYNCHRONE
# ════════════════════════════════════════════════════════════════════════════
if EXTRACTION_MODE == "sync":
    total = len(SAMPLE)
    for i, doc in enumerate(SAMPLE, 1):
        fname = doc["file"]
        path  = doc["path"]

        n = doc.get("n_pages")
        if n is None:
            try:
                with pdfplumber.open(path) as pdf:
                    n = len(pdf.pages)
            except Exception:
                n = None
            doc["n_pages"] = n

        out_path = JSON_DIR / Path(fname).with_suffix(".json").name
        if out_path.exists():
            print(f"[{i:4d}/{total}] SKIP (déjà extrait) {fname}")
            continue

        mt = max_tokens_for_pages(n)
        print(f"[{i:4d}/{total}] {fname}  ({n}p → max_tokens={mt})")

        t0 = time.time()
        try:
            with client.messages.stream(
                model=MODEL, max_tokens=mt, system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": [
                    {"type": "document", "source": {
                        "type": "base64", "media_type": "application/pdf",
                        "data": pdf_to_base64(path)}},
                    {"type": "text", "text": (
                        "Convert this financial document to JSON following the schema "
                        "defined in the system prompt. Output only valid JSON, no markdown.")}
                ]}]
            ) as stream:
                final_msg   = stream.get_final_message()
                stop_reason = final_msg.stop_reason or ""
                usage       = final_msg.usage
            block_types = [getattr(b, "type", "?") for b in (final_msg.content or [])]
            raw = "".join(
                getattr(b, "text", "") or ""
                for b in (final_msg.content or [])
                if getattr(b, "type", None) == "text"
            )
        except Exception as exc:
            print(f"  ✗ ERREUR API : {exc}")
            RESULTS[fname] = {"valid": False, "error": str(exc)}
            continue

        elapsed = round(time.time() - t0, 1)
        valid, parsed, warns = validate_json_output(raw, stop_reason)
        text_clean = normalize_model_json_response(raw)

        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(parsed, f, ensure_ascii=False, indent=2) if (valid and parsed) else f.write(text_clean)

        RESULTS[fname] = {
            "valid": valid, "parsed": parsed, "text_clean": text_clean,
            "stop_reason": stop_reason, "n_pages": n, "mt": mt, "warns": warns,
            "elapsed_s": elapsed,
            "input_tokens":  usage.input_tokens  if usage else None,
            "output_tokens": usage.output_tokens if usage else None,
        }
        status = "✓" if valid and not warns else ("⚠" if valid else "✗")
        print(f"  {status}  stop={stop_reason}  out={usage.output_tokens if usage else '?'}tok  "
              f"{elapsed}s  {'  '.join(warns) if warns else 'OK'}")
        if not text_clean:
            print(f"  ⚠ text_clean vide — blocks: {block_types}")
        time.sleep(0.5)

    print("\nExtraction synchrone terminée.")

# ════════════════════════════════════════════════════════════════════════════
#  MODE BATCH
# ════════════════════════════════════════════════════════════════════════════
else:
    if not SAMPLE:
        raise SystemExit("Rien à envoyer : augmente NB_PDF_IN_BATCH ou vide pending / extrais des JSON.")

    requests_list = []
    for doc in SAMPLE:
        path = doc["path"]
        stem = Path(path).stem
        if not BATCH_CUSTOM_ID_RE.match(stem):
            raise ValueError(f"custom_id invalide pour l'API batch : {stem!r}")

        n = doc.get("n_pages")
        if n is None:
            try:
                with pdfplumber.open(path) as pdf:
                    n = len(pdf.pages)
            except Exception:
                n = 15
            doc["n_pages"] = n

        mt      = max_tokens_for_pages(n)
        pdf_b64 = pdf_to_base64(path)
        params  = MessageCreateParamsNonStreaming(
            model=MODEL, max_tokens=mt, system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": [
                {"type": "document", "source": {
                    "type": "base64", "media_type": "application/pdf", "data": pdf_b64}},
                {"type": "text", "text": (
                    "Convert this financial document to JSON following the schema "
                    "defined in the system prompt. Output only valid JSON, no markdown.")}
            ]}]
        )
        requests_list.append(Request(custom_id=stem, params=params))
        print(f"  + {doc['file']}  ({n}p → max_tokens={mt})")

    print(f"\nCréation du batch ({len(requests_list)} requêtes)…")
    message_batch = client.messages.batches.create(requests=requests_list)
    batch_id      = message_batch.id
    print(f"batch_id = {batch_id}")
    print(f"status   = {message_batch.processing_status}")

    stems_submitted = [Path(d["path"]).stem for d in SAMPLE]
    state = load_batch_state()
    pend  = set(state.get("pending_stems", []))
    pend.update(stems_submitted)
    state["pending_stems"] = sorted(pend)
    state.setdefault("jobs", []).append({
        "batch_id": batch_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "submitted_stems": stems_submitted,
        "model": MODEL,
    })
    save_batch_state(state)
    LAST_BATCH_ID_FILE.write_text(batch_id + "\n", encoding="utf-8")
    print(f"\nBatch soumis — tu peux éteindre la machine.")
    print(f"Exécute la cellule 'Poll + import' ci-dessous quand tu reviens.")

---
## Batch : réception des résultats

*(Uniquement en mode `batch`)* Exécuter après la soumission, quand on revient.

1. Laisser `MESSAGE_BATCH_ID` vide → lecture automatique depuis `batch_last_submitted_id.txt`.
   Ou coller un id manuellement : `msgbatch_01Hkc…`
2. La cellule attend la fin du traitement, écrit les JSON dans `json_SG/` et retire les stems du fichier pending.

In [ ]:
# ── Poll + import JSON (batch) ────────────────────────────────────────────────
MESSAGE_BATCH_ID = ""  # laisser vide = lire batch_last_submitted_id.txt

if not MESSAGE_BATCH_ID.strip() and LAST_BATCH_ID_FILE.exists():
    MESSAGE_BATCH_ID = LAST_BATCH_ID_FILE.read_text(encoding="utf-8").strip()

assert MESSAGE_BATCH_ID, "Définis MESSAGE_BATCH_ID ou soumets un batch d'abord."

client = anthropic.Anthropic(api_key=API_KEY)

while True:
    batch = client.messages.batches.retrieve(MESSAGE_BATCH_ID)
    print(
        f"{batch.processing_status}  "
        f"succeeded={getattr(batch.request_counts, 'succeeded', None)}  "
        f"errored={getattr(batch.request_counts, 'errored', None)}"
    )
    if batch.processing_status == "ended":
        break
    time.sleep(POLL_INTERVAL_SEC)

released = []
errors   = []

for row in client.messages.batches.results(MESSAGE_BATCH_ID):
    cid      = row.custom_id
    out_path = JSON_DIR / f"{cid}.json"
    res      = row.result

    if res.type == "succeeded":
        msg = res.message
        raw = "".join(
            getattr(b, "text", "") or ""
            for b in (msg.content or [])
            if getattr(b, "type", None) == "text"
        )
        stop_reason          = msg.stop_reason or ""
        valid, parsed, warns = validate_json_output(raw, stop_reason)
        text_clean           = normalize_model_json_response(raw)

        if valid and parsed is not None:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(parsed, f, ensure_ascii=False, indent=2)
            released.append(cid)
            print(f"✓ {out_path.name}  OK")
        else:
            bad = out_path.with_suffix(".INVALID.json")
            with open(bad, "w", encoding="utf-8") as f:
                f.write(text_clean)
            errors.append((cid, "json_invalid", warns))
            print(f"✗ {out_path.name}  INVALID → {bad.name}")
    elif res.type == "errored":
        errors.append((cid, "errored", str(getattr(res, 'error', None))))
        print(f"✗ {cid}  API error: {getattr(res, 'error', None)}")
    else:
        errors.append((cid, res.type, ""))
        print(f"✗ {cid}  {res.type}")

if released:
    state = load_batch_state()
    pend  = set(state.get("pending_stems", []))
    pend -= set(released)
    state["pending_stems"] = sorted(pend)
    save_batch_state(state)
    print(f"\nRetirés du pending : {len(released)}  ({BATCH_STATE_PATH})")

print(f"\nTerminé. Erreurs / invalides : {len(errors)}")

In [ ]:
# ── (Optionnel) Voir l'état pending / jobs ────────────────────────────────────
load_batch_state()

---
## Analyse des résultats

*(Mode `sync` — les cellules ci-dessous explorent le dictionnaire `RESULTS` rempli lors de l'extraction synchrone.)*

In [ ]:
# ── Tableau de synthèse ───────────────────────────────────────────────────────
rows = []
for doc in SAMPLE:
    fname = doc["file"]
    r     = RESULTS.get(fname, {})
    if not r:
        continue
    idx = (r.get("parsed") or {}).get("_search_index", {})
    rows.append({
        "file":     fname[:28],
        "pages":    doc["n_pages"],
        "valid":    "✓" if r.get("valid") else "✗",
        "stop":     r.get("stop_reason", "")[:10],
        "in_tok":   r.get("input_tokens"),
        "out_tok":  r.get("output_tokens"),
        "isin_ok":  "✓" if idx.get("isin") else "✗",
        "mat_date": idx.get("maturity_date", ""),
        "undlgs":   ", ".join(idx.get("underlyings", []))[:40],
        "warnings": " | ".join(r.get("warns", [])),
    })

pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows)

In [ ]:
# ── Inspection d'un doc ───────────────────────────────────────────────────────
DOC_IDX = 0

doc   = SAMPLE[DOC_IDX]
fname = doc["file"]
r     = RESULTS.get(fname, {})

print(f"=== {fname} ===")
print(f"pages={doc['n_pages']}  valid={r.get('valid')}  stop={r.get('stop_reason')}  "
      f"in={r.get('input_tokens')}tok  out={r.get('output_tokens')}tok  {r.get('elapsed_s')}s")

if r.get("warns"):
    print("WARNINGS:", r["warns"])

parsed = r.get("parsed")
if parsed:
    print("\n── _meta ──")
    print(json.dumps(parsed.get("_meta", {}), ensure_ascii=False, indent=2))
    print("\n── _search_index ──")
    print(json.dumps(parsed.get("_search_index", {}), ensure_ascii=False, indent=2))
else:
    print("Pas de JSON parsé. text_clean (500 premiers chars):")
    print((r.get("text_clean", "") or "")[:500])

In [ ]:
# ── Formules LaTeX ────────────────────────────────────────────────────────────
def find_latex(node, results=None):
    if results is None:
        results = []
    if isinstance(node, dict):
        if "_value_latex" in node:
            results.append({"cle": node.get("_key", ""), "latex": node.get("_value_latex")})
        for v in node.values():
            find_latex(v, results)
    elif isinstance(node, list):
        for item in node:
            find_latex(item, results)
    return results


DOC_IDX_LATEX = 0
fname    = SAMPLE[DOC_IDX_LATEX]["file"]
parsed   = RESULTS.get(fname, {}).get("parsed") or {}
formules = find_latex(parsed)
print(f"Formules LaTeX dans {fname} : {len(formules)}")
for f in formules[:15]:
    print(f"  {f['cle'][:45]:<45}  {str(f['latex'])[:70]}")

In [ ]:
# ── Tableaux ──────────────────────────────────────────────────────────────────
def find_tableaux(node, results=None):
    if results is None:
        results = []
    if isinstance(node, dict):
        if node.get("_type") == "table":
            results.append({
                "titre": node.get("_title", "(sans titre)"),
                "cols":  len(node.get("_columns", [])),
                "rows":  len(node.get("_rows", [])),
            })
        for v in node.values():
            find_tableaux(v, results)
    elif isinstance(node, list):
        for item in node:
            find_tableaux(item, results)
    return results


DOC_IDX_TAB = 0
fname  = SAMPLE[DOC_IDX_TAB]["file"]
parsed = RESULTS.get(fname, {}).get("parsed") or {}
tabs   = find_tableaux(parsed)
print(f"Tableaux dans {fname} : {len(tabs)}")
for t in tabs[:20]:
    print(f"  [{t['cols']}×{t['rows']}]  {t['titre'][:70]}")

In [ ]:
# ── Restrictions & index ──────────────────────────────────────────────────────
print(f"{'Fichier':<32}  {'AssetClass':<15}  {'GovLaw':<20}  {'Listing':<25}")
print("-" * 72)
for doc in SAMPLE:
    fname = doc["file"]
    idx   = (RESULTS.get(fname, {}).get("parsed") or {}).get("_search_index", {})
    print(
        f"{fname[:30]:<32}  "
        f"{str(idx.get('asset_class', '—')):<12}  "
        f"{str(idx.get('governing_law', '—')):<12}  "
        f"{str(idx.get('listing_venue', '—')):<12}"
    )

In [ ]:
# ── Coût indicatif du run synchrone ──────────────────────────────────────────
PRICE_IN, PRICE_OUT = 1.00, 2.50  # $/M tok (ajuster selon pricing Anthropic)
total_in  = sum(r.get("input_tokens")  or 0 for r in RESULTS.values())
total_out = sum(r.get("output_tokens") or 0 for r in RESULTS.values())
print(f"Tokens input  : {total_in:,}")
print(f"Tokens output : {total_out:,}")
print(f"Coût indicatif: ${total_in/1e6*PRICE_IN + total_out/1e6*PRICE_OUT:.4f}")